## Imports

In [22]:
import pandas as pd
from google.colab import userdata, drive
from openai import OpenAI
import csv
import json
import argparse
import time
import sys
from pathlib import Path
from datetime import datetime
import os
from pydantic import BaseModel, ValidationError
from typing import Optional
from abc import ABC, abstractmethod
from openai import OpenAI
import google.generativeai as genai

In [23]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Extract Data

In [24]:
sample_validation_df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/sample_audit_extraction.csv")
all_clinical_trials_df = pd.read_csv("/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/processed_studies.csv")

## Transform

In [27]:
# @title BASE ABSTRACT CLASS
class BaseJudge(ABC):

    @abstractmethod
    def generate(self, system_prompt, user_prompt):
        pass

In [28]:
# @title OPENAI JUDGE
class OpenAIJudge(BaseJudge):

    def __init__(self, api_key, model="gpt-4.1-mini"):
        self.client = OpenAI(api_key=api_key)
        self.model = model

    def generate(self, system_prompt, user_prompt):
        response = self.client.responses.create(
            model=self.model,
            input=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            temperature=0
        )

        return response.output_text

In [29]:
class GroqJudge(BaseJudge):

    def __init__(self, api_key, model="deepseek-r1-distill-llama-70b"):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.groq.com/openai/v1"
        )
        self.model = model

    def generate(self, system_prompt, user_prompt):
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ],
                temperature=0,
            )

            if not response.choices:
                print(f"DEBUG: Groq API returned no choices. Full response: {response}")
                return ""

            content = response.choices[0].message.content
            if not content:
                print(f"DEBUG: Groq API returned empty content. Full response: {response}")
            return content
        except Exception as e:
            print(f"DEBUG: Error in GroqJudge.generate: {e}")
            return ""

In [30]:
# @title GEMINI JUDGE
class GeminiJudge(BaseJudge):

    def __init__(self, api_key, model="gemini-2.5-flash"):
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model)

    def generate(self, system_prompt, user_prompt):
        full_prompt = f"""
          {system_prompt}
          {user_prompt}
        """

        response = self.model.generate_content(
            full_prompt,
            generation_config={"temperature": 0}
        )

        return response.text

In [31]:
# @title CLINICAL AUDITOR
class ClinicalExtractionAuditor:

    def __init__(self, judge, output_path="audit_results.csv"):
        self.judge = judge
        self.output_path = output_path
        self.system_prompt = """
            You are a Senior Clinical Data Auditor specializing in clinical trial eligibility protocols.

            You must evaluate whether extracted inclusion and exclusion criteria are faithful to the original eligibility criteria text.

            You must identify:
            - omissions
            - hallucinations
            - logical distortions
            - formatting issues

            You must respond ONLY with valid JSON.
            """

    def _generate_user_prompt(self, raw_text, inclusion, exclusion):

        return f"""
          RAW ELIGIBILITY CRITERIA:
          {raw_text}

          EXTRACTED INCLUSION CRITERIA:
          {inclusion}

          EXTRACTED EXCLUSION CRITERIA:
          {exclusion}

          EVALUATION RUBRIC:

          5 = perfect extraction
          4 = minor formatting issue
          3 = minor omission
          2 = major omission or logical distortion
          1 = severe failure

          Return ONLY valid JSON:

          {{
            "judge_score": integer,
            "tp": integer,
            "fp": integer,
            "fn": integer,
            "justification": string,
            "main_category": "Omission" | "Distortion" | "Hallucination" | "None"
          }}
        """

    def audit_case(self, nct_id, raw_text, inclusion, exclusion):
        try:
            response_text = self.judge.generate(
                system_prompt=self.system_prompt,
                user_prompt=self._generate_user_prompt(
                    raw_text,
                    inclusion,
                    exclusion
                )
            )

            if not response_text.strip():
                print(f"ALERTA: Resposta vazia recebida para {nct_id}")

            # limpeza opcional para modelos que retornam markdown
            response_text = response_text.replace(
                "```json",
                ""
            ).replace(
                "```",
                ""
            ).strip()

            if "<think>" in response_text:
                response_text = response_text.split("</think>")[-1].strip()

            parsed = json.loads(response_text)

            parsed["nct_id"] = nct_id

            return parsed

        except Exception as e:
            print(f"Error auditing {nct_id}: {e}")

            # RATE LIMIT
            error_message = str(e)
            if "rate_limit_exceeded" in error_message:
              wait_time = 15
              print(
                  f"Rate limit reached for {nct_id}. "
                  f"Waiting {wait_time}s..."
              )
              time.sleep(wait_time)
              # continue

            return {
                "nct_id": nct_id,
                "judge_score": None,
                "tp": None,
                "fp": None,
                "fn": None,
                "justification": str(e),
                "main_category": "None"
            }

    def run_batch(self, df_input):
      required_cols = [
          "nct_id",
          "eligibility_criteria",
          "inclusion_criteria",
          "exclusion_criteria"
      ]

      missing_cols = [
          col for col in required_cols
          if col not in df_input.columns
      ]

      if missing_cols:
          raise ValueError(f"Missing required columns: {missing_cols}")

      # LOAD EXISTING RESULTS
      processed_ids = set()
      if os.path.exists(self.output_path):

          existing_df = pd.read_csv(
              self.output_path
          )

          processed_ids = set(
              existing_df["nct_id"]
          )

          print(
              f"Loaded {len(processed_ids)} "
              f"already processed studies."
          )

      total = len(df_input)

      for idx, row in df_input.iterrows():

          nct_id = row["nct_id"]

          # SKIP ALREADY PROCESSED
          if nct_id in processed_ids:

              print(
                  f"[{idx+1}/{total}] "
                  f"Skipping {nct_id}"
              )

              continue

          print(
              f"[{idx+1}/{total}] "
              f"Auditing {nct_id}"
          )

          result = self.audit_case(
              nct_id=nct_id,
              raw_text=row["eligibility_criteria"],
              inclusion=row["inclusion_criteria"],
              exclusion=row["exclusion_criteria"]
          )

          # APPEND IMMEDIATELY
          result_df = pd.DataFrame([result])

          result_df.to_csv(
              self.output_path,
              mode="a",
              header=not os.path.exists(self.output_path),
              index=False
          )

          # SMALL DELAY
          time.sleep(3)

      print("\nAudit completed.")

## Main

In [34]:
if __name__ == "__main__":

    # ======================================
    # OPTION 1 — OPENAI
    # ======================================

    # judge = OpenAIJudge(
    #     api_key=userdata.get("JudgeOpenAiAPI"),
    #     model="gpt-4.1-mini"
    # )

    # ======================================
    # OPTION 2 — GROQ
    # ======================================

    judge = GroqJudge(
        api_key=userdata.get("JudgeGROQAPI"),
        model="llama-3.3-70b-versatile"
    )

    # ======================================
    # OPTION 3 — GEMINI
    # ======================================

    # judge = GeminiJudge(
    #     api_key=userdata.get("ParsingGeminiAPI"),
    #     model="gemini-2.5-flash"
    # )

    # sample_auditor = ClinicalExtractionAuditor(
    #     judge=judge,
    #     output_path="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/audit_sample_extraction_result.csv"
    # )
    # sample_results = sample_auditor.run_batch(sample_validation_df)

    full_auditor = ClinicalExtractionAuditor(
        judge=judge,
        output_path="/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/audit_extraction_result.csv"
    )

    full_results = full_auditor.run_batch(all_clinical_trials_df)

Loaded 996 already processed studies.
[28/5] Auditing NCT02157506
[81/5] Auditing NCT00851877
[484/5] Auditing NCT01104558
[780/5] Auditing NCT02281760
[963/5] Auditing NCT02481466

Audit completed.
